# VQA Múa Lân — Train & Eval (A1, A2, A3)

Huấn luyện và đánh giá **3 cấu hình decoder** trên cùng pipeline:

| Run | Decoder       | Norm      | FFN        | Ghi chú |
|-----|---------------|-----------|------------|---------|
| **A1** | LSTM        | —         | —          | Baseline rời (RNN cổ điển) |
| **A2** | Transformer | LayerNorm | VanillaFFN | Transformer cổ điển (Attention Is All You Need) |
| **A3** | Transformer | RMSNorm   | SwiGLU     | Modern stack (LLaMA / Gemma / Qwen) |

Encoder cố định: SigLIP2-B/16 (layer −2, frozen) + PhoBERT-v2 (mean 4 last layers, frozen) + Cross-Attention Fusion.
Loss: `CrossEntropy(ignore_index=PAD, label_smoothing=0.1)`.

Notebook đặt ở project root; mọi class import trực tiếp từ `src/` — không lặp code.

## 1. Setup

In [ ]:
import os, sys, time, json, random
import numpy as np
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import ModelConfig, TrainConfig
from src.build  import (build_tokenizer_and_processor, resolve_special_ids,
                        build_loaders, build_model)
from src.training import Trainer, Evaluator
from src.metrics  import ExactMatch, BLEUScore, METEORScore

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('project_root =', PROJECT_ROOT)
print('device       =', DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## 2. Tokenizer & DataLoaders (dùng chung cho 3 run)

In [ ]:
_probe_cfg = ModelConfig()
tokenizer, image_processor = build_tokenizer_and_processor(_probe_cfg)
_probe_cfg = resolve_special_ids(tokenizer, _probe_cfg)

BATCH_SIZE = 16
EPOCHS     = 8
LR         = 3e-4

# Đường dẫn dữ liệu phải truyền tường minh — TrainConfig không có default cho mục này.
TRAIN_JSON = os.path.join(PROJECT_ROOT, 'qa_data', 'train.json')
VAL_JSON   = os.path.join(PROJECT_ROOT, 'qa_data', 'val.json')
TEST_JSON  = os.path.join(PROJECT_ROOT, 'qa_data', 'test.json')
IMAGE_ROOT = PROJECT_ROOT

shared_train_cfg = TrainConfig(
    train_json=TRAIN_JSON, val_json=VAL_JSON, test_json=TEST_JSON,
    image_root=IMAGE_ROOT,
    batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, num_workers=0,
    ckpt_dir=os.path.join(PROJECT_ROOT, 'checkpoints'),
    log_dir=os.path.join(PROJECT_ROOT, 'logs'),
    eval_every=1, save_every=EPOCHS,
)
train_loader, val_loader, test_loader = build_loaders(
    _probe_cfg, shared_train_cfg, tokenizer, image_processor)
print(f'train={len(train_loader.dataset)}  val={len(val_loader.dataset)}  test={len(test_loader.dataset)}')

## 3. Trainer-with-history

Bọc lại `Trainer` để lưu `train_loss` và mọi metric val theo từng epoch — phục vụ vẽ biểu đồ line.

In [ ]:
class TrainerWithHistory(Trainer):
    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.history = {'train_loss': [], 'val': {}}
    def fit(self):
        cfg = self.train_cfg
        for epoch in range(cfg.epochs):
            t0 = time.time()
            tl = self.train_one_epoch(epoch)
            self.history['train_loss'].append(tl)
            log = f'epoch={epoch+1}/{cfg.epochs} train_loss={tl:.4f} time={time.time()-t0:.1f}s'
            if self.evaluator is not None and self.val_loader is not None:
                res = self.evaluator.evaluate(self.model, self.val_loader)
                for k, v in res.items():
                    self.history['val'].setdefault(k, []).append(v)
                log += '  ' + ' '.join(f'val_{k}={v:.4f}' for k, v in res.items())
            self._log(log)
        self._save_ckpt('final')
        return self.history

In [ ]:
def run_config(tag, decoder_type, norm_type='layernorm', ffn_type='vanilla'):
    """`norm_type` / `ffn_type` chỉ ảnh hưởng khi decoder_type='transformer';
    với 'lstm' chúng bị bỏ qua nên có thể không truyền."""
    print(f'\n========== {tag} ==========')
    mcfg = ModelConfig(decoder_type=decoder_type, norm_type=norm_type, ffn_type=ffn_type)
    mcfg = resolve_special_ids(tokenizer, mcfg)
    tcfg = TrainConfig(
        train_json=TRAIN_JSON, val_json=VAL_JSON, test_json=TEST_JSON,
        image_root=IMAGE_ROOT,
        batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR, num_workers=0,
        ckpt_dir=os.path.join(PROJECT_ROOT, 'checkpoints'),
        log_dir=os.path.join(PROJECT_ROOT, 'logs'),
        eval_every=1, save_every=EPOCHS, run_name=tag,
    )
    model = build_model(mcfg)
    metrics = [ExactMatch(), BLEUScore(), METEORScore()]
    evaluator = Evaluator(
        tokenizer=tokenizer, metrics=metrics,
        bos_id=mcfg.bos_id, eos_id=mcfg.eos_id, pad_id=mcfg.pad_id,
        max_len=mcfg.max_answer_len, beam_size=1,
    )
    trainer = TrainerWithHistory(
        model=model, train_loader=train_loader, val_loader=val_loader,
        train_cfg=tcfg, model_cfg=mcfg, evaluator=evaluator,
    )
    history = trainer.fit()
    return history, trainer, evaluator, model

## 4. Train A1 — LSTM

In [ ]:
hist_A1, trainer_A1, evaluator_A1, model_A1 = run_config(
    'A1_lstm', decoder_type='lstm')

## 5. Train A2 — Transformer + LayerNorm + VanillaFFN

In [ ]:
hist_A2, trainer_A2, evaluator_A2, model_A2 = run_config(
    'A2_transformer_vanilla', decoder_type='transformer', norm_type='layernorm', ffn_type='vanilla')

## 6. Train A3 — Transformer + RMSNorm + SwiGLU

In [ ]:
hist_A3, trainer_A3, evaluator_A3, model_A3 = run_config(
    'A3_transformer_modern', decoder_type='transformer', norm_type='rmsnorm', ffn_type='swiglu')

## 7. So sánh — biểu đồ line

In [ ]:
def plot_compare(histories, keys, title_prefix=''):
    n = len(keys)
    fig, axes = plt.subplots(1, n, figsize=(5.2*n, 4))
    if n == 1: axes = [axes]
    for ax, key in zip(axes, keys):
        for tag, h in histories.items():
            ys = h['train_loss'] if key == 'train_loss' else h['val'].get(key, [])
            xs = list(range(1, len(ys)+1))
            ax.plot(xs, ys, marker='o', label=tag)
        ax.set_title(f'{title_prefix}{key}')
        ax.set_xlabel('epoch'); ax.grid(True, alpha=0.3); ax.legend()
    plt.tight_layout(); plt.show()

histories = {
    'A1_lstm':                hist_A1,
    'A2_transformer_vanilla': hist_A2,
    'A3_transformer_modern':  hist_A3,
}
plot_compare(histories, ['train_loss'])
plot_compare(histories, ['exact_match', 'bleu', 'meteor'], title_prefix='val_')

## 8. Đánh giá cuối trên test split (Exact Match, BLEU, METEOR)

In [ ]:
test_results = {}
for tag, model in [('A1_lstm', model_A1),
                   ('A2_transformer_vanilla', model_A2),
                   ('A3_transformer_modern',  model_A3)]:
    metrics = [ExactMatch(), BLEUScore(), METEORScore()]
    ev = Evaluator(
        tokenizer=tokenizer, metrics=metrics,
        bos_id=_probe_cfg.bos_id, eos_id=_probe_cfg.eos_id, pad_id=_probe_cfg.pad_id,
        max_len=_probe_cfg.max_answer_len, beam_size=1,
    )
    test_results[tag] = ev.evaluate(model, test_loader)
    print(tag, test_results[tag])

## 9. Lưu lại history + test results

In [ ]:
log_dir = os.path.join(PROJECT_ROOT, 'logs')
os.makedirs(log_dir, exist_ok=True)
with open(os.path.join(log_dir, 'train_eval_A_history.json'), 'w', encoding='utf-8') as f:
    json.dump({'history': histories, 'test': test_results}, f, ensure_ascii=False, indent=2)
print('saved -> logs/train_eval_A_history.json')